# 🕵️‍♂️ Pakistani Social Media Content Detective
## Step 2: Collect YouTube videos and comments

**Before you start:**
- **On Google Colab:** add your YouTube API key in the 🔑 *Secrets* panel (left sidebar) with the name `YOUTUBE_API_KEY` and switch on notebook access.
- **On your computer (VS Code):**
  1. Install the packages once: open a terminal in this folder and run `python3 -m pip install -r requirements.txt`
  2. Open the `.env` file in this folder and paste your key after `YOUTUBE_API_KEY=`.

Then run each cell in order (click ▶ on each one), or use **Run All**.

### Cell 1: Connect to YouTube

In [1]:
import os
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
import pandas as pd

# On Colab the key comes from Secrets; on your computer it comes from the .env file
try:
    from google.colab import userdata
    IN_COLAB = True
    API_KEY = userdata.get("YOUTUBE_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    IN_COLAB = False
    load_dotenv(".env")
    API_KEY = os.getenv("YOUTUBE_API_KEY")

if not API_KEY:
    raise ValueError("No API key found. Paste your key into the .env file after YOUTUBE_API_KEY=")

youtube = build("youtube", "v3", developerKey=API_KEY)
print("Connected!")


[notice] A new release of pip is available: 25.0 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Connected!


### Cell 2: Your settings (the ONLY cell you need to edit)
Find the brand's handle: open their YouTube channel, and under the channel name you'll see something like `@brandname`. Paste it below.

In [2]:
CHANNEL_HANDLE = "@PakWheels"
MAX_VIDEOS = 1000
MAX_COMMENTS_PER_VIDEO = 200
REVIEW_KEYWORDS = ["review", "test drive", "first drive"]
MIN_COMMENTS = 10

### Cell 3: Find the channel

In [3]:
response = youtube.channels().list(
    part="snippet,contentDetails,statistics",
    forHandle=CHANNEL_HANDLE
).execute()

if not response.get("items"):
    raise ValueError("Channel not found. Check the handle in Cell 2 (it must start with @).")

channel = response["items"][0]
uploads_playlist = channel["contentDetails"]["relatedPlaylists"]["uploads"]
print("Found channel:", channel["snippet"]["title"])
print("Subscribers:", channel["statistics"].get("subscriberCount", "hidden"))
print("Total videos:", channel["statistics"].get("videoCount"))

Found channel: PakWheels.com
Subscribers: 2860000
Total videos: 11462


### Cell 4: Collect the videos (title, views, likes, comment count)

In [4]:
# 1) Get the IDs of the channel's latest videos
video_ids = []
next_page = None
while len(video_ids) < MAX_VIDEOS:
    res = youtube.playlistItems().list(
        part="contentDetails", playlistId=uploads_playlist,
        maxResults=50, pageToken=next_page
    ).execute()
    for item in res["items"]:
        video_ids.append(item["contentDetails"]["videoId"])
    next_page = res.get("nextPageToken")
    if not next_page:
        break
video_ids = video_ids[:MAX_VIDEOS]

# 2) Get details for each video
videos = []
for i in range(0, len(video_ids), 50):
    res = youtube.videos().list(
        part="snippet,statistics", id=",".join(video_ids[i:i+50])
    ).execute()
    for v in res["items"]:
        stats = v["statistics"]
        videos.append({
            "video_id": v["id"],
            "title": v["snippet"]["title"],
            "published": v["snippet"]["publishedAt"],
            "views": int(stats.get("viewCount", 0)),
            "likes": int(stats.get("likeCount", 0)),
            "comment_count": int(stats.get("commentCount", 0)),
        })

videos_df = pd.DataFrame(videos)
print(f"Collected {len(videos_df)} videos")
is_review = videos_df["title"].str.lower().str.contains("|".join(REVIEW_KEYWORDS))
enough_comments = videos_df["comment_count"] >= MIN_COMMENTS
videos_df = videos_df[is_review & enough_comments].reset_index(drop=True)
print(f"Kept {len(videos_df)} review videos")
print("Total comments available:", videos_df["comment_count"].sum())
videos_df.head()

Collected 1000 videos
Kept 114 review videos
Total comments available: 13557


,video_id,title,published,views,likes,comment_count
0,ziOGUS2GnI0,iCAUR V27 | Test Drive Review Pakwheels,2026-10-23T14:00:06Z,65913,748,91
1,Wwq--FyoCg0,"Chery Tiggo 8 PHEV | 11,000 KM Driven, EV Ran...",2026-10-22T14:27:06Z,72085,842,83
2,Im3gKdlRYlY,"BYD Atto 2 | 2026 Model, Comfort, Features & M...",2026-10-21T14:00:10Z,206531,1884,88
3,2jpq8VQBxuE,"Chery Tiggo 9 | 600+ HP, 20-Inch Rims & Real ...",2026-10-20T14:00:02Z,66145,573,36
4,rZ6dHrbbLPU,"Ducky Bhai Jetour T2 | Features, EV Range & Dr...",2026-10-19T15:28:07Z,876363,8145,259


### Cell 5: Collect the comments (this may take a few minutes)

In [7]:
all_comments = []

for n, vid in enumerate(videos_df["video_id"], start=1):
    next_page = None
    count = 0
    try:
        while count < MAX_COMMENTS_PER_VIDEO:
            res = youtube.commentThreads().list(
                part="snippet", videoId=vid, maxResults=100,
                pageToken=next_page, textFormat="plainText"
            ).execute()
            for item in res["items"]:
                c = item["snippet"]["topLevelComment"]["snippet"]
                all_comments.append({
                    "video_id": vid,
                    "comment": c["textDisplay"],
                    "comment_likes": c.get("likeCount", 0),
                    "replies": item["snippet"].get("totalReplyCount", 0),
                    "published": c["publishedAt"],
                })
                count += 1
            next_page = res.get("nextPageToken")
            if not next_page:
                break
    except HttpError:
        pass  # comments are turned off on this video, so skip it

    if n % 10 == 0:
        print(f"{n} videos done, {len(all_comments)} comments so far")

comments_df = pd.DataFrame(all_comments)
print(f"\nDONE! Total comments collected: {len(comments_df)}")
if len(comments_df) < 2000:
    print("Less than 2,000 comments. Increase MAX_VIDEOS in Cell 2, or pick a busier brand.")
comments_df.head(10)

10 videos done, 856 comments so far
20 videos done, 1517 comments so far
30 videos done, 2302 comments so far
40 videos done, 2701 comments so far
50 videos done, 3485 comments so far
60 videos done, 4018 comments so far
70 videos done, 4538 comments so far
80 videos done, 5462 comments so far
90 videos done, 6257 comments so far
100 videos done, 7404 comments so far
110 videos done, 7950 comments so far

DONE! Total comments collected: 8460


,video_id,comment,comment_likes,replies,published
0,ziOGUS2GnI0,"wo sub to thk hai , front number plate k ""E"" k...",0,0,2026-10-24T08:33:33Z
1,ziOGUS2GnI0,Cherry Tico9. VS Fortuner,0,0,2026-10-24T06:25:09Z
2,ziOGUS2GnI0,Defender Copy,0,0,2026-10-24T06:05:06Z
3,ziOGUS2GnI0,Nice,0,0,2026-10-24T04:54:47Z
4,ziOGUS2GnI0,"for the price, good entry ... LC200 now appear...",0,0,2026-10-24T03:54:35Z
5,ziOGUS2GnI0,Well paid.,1,0,2026-10-24T01:57:44Z
6,ziOGUS2GnI0,"Bhai bura na many tou aik baat bolu, yee ghari...",1,1,2026-10-23T23:21:15Z
7,ziOGUS2GnI0,Paid promotion,0,0,2026-10-23T23:07:26Z
8,ziOGUS2GnI0,defender copy,0,0,2026-10-23T21:08:55Z
9,ziOGUS2GnI0,"Ye sahib bilkul sahi keh rahy hein, mujhy bhi ...",0,0,2026-10-23T21:07:53Z


### Cell 6: Save and download your data

In [8]:
videos_df.to_csv("videos.csv", index=False, encoding="utf-8-sig")
comments_df.to_csv("comments.csv", index=False, encoding="utf-8-sig")

# On Colab, download the files. On your computer they are already saved in this folder.
if IN_COLAB:
    from google.colab import files
    files.download("videos.csv")
    files.download("comments.csv")
print("Saved! Keep these two files safe; we'll use them in Step 3.")

Saved! Keep these two files safe; we'll use them in Step 3.
